# D009 | 빵셀러 데이터 랭글링 종합 프로젝트

**AIFFEL AI 데이터 인텔리전스 2기 · 데이터 랭글링 종합 실습 · 2026-10-27**

**분석 목적:** 공식 실습 노트북의 가상 빵셀러 운영 데이터를 이용해 **데이터 읽기 → 품질 진단 → 정제 → 검증 → KPI 집계 → Parquet 저장 → 판단 근거 문서화**를 완성한다.

- **데이터 출처:** 사용자가 제공한 AIFFEL LMS 노트북의 빵셀러 데이터 생성 셀을 실행해 얻은 `data/bakery.csv` (가상·합성 데이터). 노드 원문·코드는 제출 노트북에 복제하지 않았다.
- **AI 사용:** 코드 구조화, 검증 코드 작성, 문서 초안 작성에 AI의 도움을 받음. 실제 데이터·출력·업무 규칙의 적절성은 학습자가 검토해야 한다.
- **주의:** 합성 데이터에는 거래 ID가 없다. 동일 행을 실습 규칙에 따라 제거하되 **서로 다른 실제 거래인지 독립 검증 불가**라는 한계를 명시한다. 공개 저장소에 데이터 CSV를 올릴 수 있는지는 수업 규정을 확인한다.
- **실행 순서:** `Kernel → Restart & Run All` 또는 `런타임 → 모두 실행`. `pyarrow`가 없다면 `%pip install pyarrow` 실행 후 커널 재시작/전체 재실행.

In [1]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


## 0. 인제스트(Ingest) — 원본 파일 읽기

**왜 필요한가?** 입력 파일을 읽은 다음 행 수, 열 이름, 자료형이 예상과 같은지 확인하고 원본과 정제본을 구분하기 위해서다. 입력은 원본 CSV, 출력은 판다스 데이터프레임(DataFrame)이다.

In [2]:
from pathlib import Path
import sys
import importlib.util
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# D009 폴더 내부에서 실행하는 것을 기본으로 하고 저장소 루트 실행도 지원
BASE = Path.cwd()
if not (BASE / 'data' / 'bakery.csv').exists() and (BASE / 'D009' / 'data' / 'bakery.csv').exists():
    BASE = BASE / 'D009'
RAW_PATH = BASE / 'data' / 'bakery.csv'
OUT_DIR = BASE / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
if not RAW_PATH.exists():
    raise FileNotFoundError(f'필수 원본 CSV가 없습니다: {RAW_PATH.resolve()}')

bakery_raw = pd.read_csv(RAW_PATH)
required = {'store_id', 'date_str', 'item', 'quantity', 'unit_price', 'revenue'}
assert required <= set(bakery_raw.columns), f'필수 열 누락: {required - set(bakery_raw.columns)}'
print('Python:',sys.version.split()[0], 'pandas:',pd.__version__)
print('공식 실습 생성 CSV:',RAW_PATH.resolve())
print('원본 형태:',bakery_raw.shape)
display(bakery_raw.head())
print('원본 자료형:'); display(bakery_raw.dtypes.rename('dtype').to_frame())
print('Parquet 엔진 설치 여부:', '설치됨' if importlib.util.find_spec('pyarrow') else 'pyarrow 미설치')

Python: 3.12.15 pandas: 3.0.6
공식 실습 생성 CSV: C:\Users\shini\Downloads\D009_%EC%98%A4%EB%8A%98%EC%A0%9C%EC%B6%9C_%EC%A4%80%EB%B9%84%EB%AC%B6%EC%9D%8C\D009\data\bakery.csv
원본 형태: (2164, 6)


,store_id,date_str,item,quantity,unit_price,revenue
0,S01,20250401,쿠키,2,12000,24000.0
1,S02,2025-04-01,케이크,2,12000,24000.0
2,S03,2025-04-01,쿠키,1,5500,5500.0
3,S04,2025-06-20,쿠키,2,8500,17000.0
4,S05,20250401,샌드위치,5,4500,22500.0


원본 자료형:


,dtype
store_id,str
date_str,str
item,str
quantity,int64
unit_price,int64
revenue,float64


Parquet 엔진 설치 여부: 설치됨


## 1. 품질 진단 — `quality_report_full()`

**검사 항목:** 행·열 수, 자료형, 결측 수·비율, 완전 동일 행, 고유값, 수치형 IQR 이상치 비율, 날짜로 해석 가능한 열. `quality_report_full()`은 **데이터를 변경하지 않고** 진단표만 만든다.

In [3]:
def quality_report_full(df: pd.DataFrame, name: str = 'data') -> pd.DataFrame:
    """반복 사용 가능한 결측·중복·수치형 이상치·자료형 진단."""
    print(f'[{name}] 행·열: {df.shape}, 메모리: {df.memory_usage(deep=True).sum():,} bytes, 완전 동일 행: {df.duplicated().sum():,}')
    rows=[]
    for col in df.columns:
        s=df[col]
        missing_pct=100*s.isna().mean() if len(df) else 0
        outlier_pct=np.nan
        if pd.api.types.is_numeric_dtype(s):
            vals=s.dropna()
            if len(vals):
                q1,q3=vals.quantile([.25,.75]); iqr=q3-q1
                if iqr > 0:
                    outlier_pct=100*((vals<q1-1.5*iqr)|(vals>q3+1.5*iqr)).mean()
                else:
                    outlier_pct=100*(vals != q1).mean()
        # 날짜라는 의미가 분명한 컬럼에만 파싱 가능성을 점검
        maybe_date=False
        if 'date' in col.lower() and (pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s)):
            sample=s.dropna().astype(str).head(50)
            if len(sample):
                parsed=pd.to_datetime(sample,format='mixed',errors='coerce')
                maybe_date=bool(parsed.notna().mean() > 0.8)
        rows.append({'column':col,'dtype':str(s.dtype),'missing_n':int(s.isna().sum()),
            'missing_pct':round(missing_pct,2),'unique_n':int(s.nunique(dropna=True)),
            'outlier_pct_iqr':round(outlier_pct,2) if pd.notna(outlier_pct) else np.nan,
            'maybe_datetime':maybe_date})
    return pd.DataFrame(rows)

quality_before=quality_report_full(bakery_raw,'bakery_raw')
display(quality_before)
print('결측률 상위:',quality_before.sort_values('missing_pct',ascending=False).head(3)[['column','missing_pct']].to_dict('records'))
print('수치형 IQR 이상치 비율 상위:',quality_before.dropna(subset=['outlier_pct_iqr']).sort_values('outlier_pct_iqr',ascending=False)[['column','outlier_pct_iqr']].to_dict('records'))
print('날짜 후보:',quality_before.loc[quality_before['maybe_datetime'],'column'].tolist())

[bakery_raw] 행·열: (2164, 6), 메모리: 150,045 bytes, 완전 동일 행: 321


,column,dtype,missing_n,missing_pct,unique_n,outlier_pct_iqr,maybe_datetime
0,store_id,str,0,0.00,7,NaN,False
1,date_str,str,0,0.00,9,NaN,True
2,item,str,0,0.00,6,NaN,False
3,quantity,int64,0,0.00,5,14.56,False
4,unit_price,int64,0,0.00,5,0.00,False
5,revenue,float64,60,2.77,26,17.30,False


결측률 상위: [{'column': 'revenue', 'missing_pct': 2.77}, {'column': 'store_id', 'missing_pct': 0.0}, {'column': 'date_str', 'missing_pct': 0.0}]
수치형 IQR 이상치 비율 상위: [{'column': 'revenue', 'outlier_pct_iqr': 17.3}, {'column': 'quantity', 'outlier_pct_iqr': 14.56}, {'column': 'unit_price', 'outlier_pct_iqr': 0.0}]
날짜 후보: ['date_str']


## 2. 정제(Clean) — 재사용 가능한 작은 함수와 `.pipe()`

**정제 규칙 및 근거:** ① 완전 동일 행 제거(과제 가정), ② 매장 ID 공백 제거, ③ `케익`→`케이크` 표준화, ④ 혼합 날짜 파싱, ⑤ 음수 매출은 환불 의심 기록으로 **별도 보존**, ⑥ 매출 결측은 총매출을 임의 추정하지 않기 위해 분석대상에서 제외.

**유의:** 원본 완전 동일 행이 매우 많지만 실제 거래 ID가 없으므로 서로 다른 거래인지 판단할 수 없다. 이는 교육용 가정이다. 문자열·날짜 표준화 이후 새로 동일해지는 행은 무조건 추가 삭제하지 않는다. 수량 50은 이상치 후보이지만 실제 대량주문일 가능성이 있어 자동 삭제하지 않는다.

In [4]:
def step_deduplicate(df):
    return df.drop_duplicates(keep='first').copy()

def step_store_id(df):
    return df.assign(store_id=df['store_id'].astype('string').str.strip())

def step_item_name(df):
    return df.assign(item=df['item'].astype('string').str.strip().replace({'케익':'케이크'}))

def step_parse(df):
    return df.assign(date=pd.to_datetime(df['date_str'],format='mixed',errors='coerce'),
                     revenue=pd.to_numeric(df['revenue'],errors='coerce'))

def step_refund_exclusion(df):
    return df.loc[~df['revenue'].lt(0)].copy()

def step_missing_revenue(df):
    return df.dropna(subset=['revenue']).copy()

def preprocess_bakery(df):
    # 원본 보존 + 여섯 단계 .pipe 체이닝
    src=df.copy(deep=True)
    steps=[('원본',src)]
    a=src.pipe(step_deduplicate); steps.append(('1. 완전 동일 행 제거',a))
    b=a.pipe(step_store_id); steps.append(('2. 매장 ID 정제',b))
    c=b.pipe(step_item_name); steps.append(('3. 상품명 표준화',c))
    d=c.pipe(step_parse); steps.append(('4. 날짜·매출 자료형 변환',d))
    refunds=d.loc[d['revenue'].lt(0)].copy()
    e=d.pipe(step_refund_exclusion); steps.append(('5. 음수매출 별도 보관',e))
    f=e.pipe(step_missing_revenue).reset_index(drop=True); steps.append(('6. 매출 결측 제거',f))
    stage=pd.DataFrame({'단계':[x for x,_ in steps], '행_수':[len(y) for _,y in steps]})
    stage['전단계대비_제외']=(-stage['행_수'].diff()).fillna(0).astype(int)
    return f, refunds, d, stage

bakery_clean, refunds_bakery, bakery_prepared, stage_log = preprocess_bakery(bakery_raw)
print('정제 단계별 기록:'); display(stage_log)
print('정제 후:',bakery_clean.shape, '/ 환불 의심 별도 보관:',len(refunds_bakery))
display(bakery_clean.head())

정제 단계별 기록:


,단계,행_수,전단계대비_제외
0,원본,2164,0
1,1. 완전 동일 행 제거,1843,321
2,2. 매장 ID 정제,1843,0
3,3. 상품명 표준화,1843,0
4,4. 날짜·매출 자료형 변환,1843,0
5,5. 음수매출 별도 보관,1842,1
6,6. 매출 결측 제거,1784,58


정제 후: (1784, 7) / 환불 의심 별도 보관: 1


,store_id,date_str,item,quantity,unit_price,revenue,date
0,S01,20250401,쿠키,2,12000,24000.0,2025-04-01
1,S02,2025-04-01,케이크,2,12000,24000.0,2025-04-01
2,S03,2025-04-01,쿠키,1,5500,5500.0,2025-04-01
3,S04,2025-06-20,쿠키,2,8500,17000.0,2025-06-20
4,S05,20250401,샌드위치,5,4500,22500.0,2025-04-01


## 3. 검증(Validation) — 숫자가 조용히 사라지거나 늘어나지 않았는가?

`assert`로 필수 규칙을 확인한다. **중요:** 표기 통일 후 서로 다른 원본 기록이 동일하게 보일 수 있으므로, 최종 동일 행이 있다고 다시 제거하지 않는다. 원본 행 수는 `정제 후 + 완전중복 제거 + 환불 분리 + 매출 결측 제외`로 재조정한다.

In [5]:
counts=stage_log.set_index('단계')['전단계대비_제외']
dedup_n=int(counts['1. 완전 동일 행 제거'])
refund_n=int(counts['5. 음수매출 별도 보관'])
missing_n=int(counts['6. 매출 결측 제거'])
parse_fail= int((bakery_prepared['date_str'].notna() & bakery_prepared['date'].isna()).sum())

assert len(bakery_clean)+dedup_n+refund_n+missing_n==len(bakery_raw), '행 수 재조정 실패'
assert bakery_clean['revenue'].notna().all(), '정제 대상 매출 결측 발생'
assert bakery_clean['revenue'].ge(0).all(), '환불 의심 음수 매출 유입'
assert bakery_clean['date'].notna().all(), '날짜 파싱 실패가 월별 KPI를 누락시킬 위험'
assert bakery_clean['store_id'].str.strip().eq(bakery_clean['store_id']).all(), '매장 ID 공백 미정제'
assert (~bakery_clean['item'].eq('케익')).all(), '상품명 표기 혼재 미해결'
print(f'원본 {len(bakery_raw):,}행 = 정제 {len(bakery_clean):,} + 원본 중복 {dedup_n:,} + 환불 {refund_n:,} + 매출 결측 {missing_n:,}')
print('원래 값이 있었으나 날짜 변환에 실패:',parse_fail, '행')
print('표준화 이후 새로 동일하게 보이는 행 (재삭제하지 않음):',int(bakery_clean.duplicated().sum()))
print('수량=50 관측 행(이상치 후보, 유지):', int(bakery_clean['quantity'].eq(50).sum()))
print('원본에서 관측된 날짜의 종류:',sorted(bakery_clean['date'].dt.strftime('%Y-%m-%d').unique()))
print('검증 통과: 행 수 보존·환불 분리·결측·문자열·날짜 규칙')

원본 2,164행 = 정제 1,784 + 원본 중복 321 + 환불 1 + 매출 결측 58
원래 값이 있었으나 날짜 변환에 실패: 0 행
표준화 이후 새로 동일하게 보이는 행 (재삭제하지 않음): 5
수량=50 관측 행(이상치 후보, 유지): 277
원본에서 관측된 날짜의 종류: ['2025-04-01', '2025-05-15', '2025-06-20']
검증 통과: 행 수 보존·환불 분리·결측·문자열·날짜 규칙


## 4. KPI 변환(Transform) — 두 가지 필수 분석표

- **월별·매장별 매출:** 행=매장, 열=월, 값=매출 합계. 원본의 `date_str`은 날짜로 바꾼 뒤 연·월을 추출한다.
- **상품별 KPI:** 상품별 평균 단가=`unit_price`의 단순 평균, 총매출=`revenue`의 합계, 기록 건수. 평균 단가는 판매 수량 가중평균과 다르므로 의미를 명시한다.

In [6]:
sales=bakery_clean.copy()
sales['month']=sales['date'].dt.to_period('M').astype(str)
store_month=(sales.pivot_table(index='store_id', columns='month', values='revenue', aggfunc='sum',fill_value=0)
             .sort_index().reset_index())
store_month.columns.name=None
item_kpi=(sales.groupby('item',dropna=False,as_index=False)
          .agg(avg_unit_price=('unit_price','mean'),total_revenue=('revenue','sum'),n_rows=('revenue','size'))
          .sort_values('total_revenue',ascending=False).reset_index(drop=True))

store_revenue=float(store_month.select_dtypes(include='number').sum().sum())
item_revenue=float(item_kpi['total_revenue'].sum())
source_revenue=float(sales['revenue'].sum())
assert np.isclose(store_revenue,source_revenue), '월별·매장별 매출 검산 실패'
assert np.isclose(item_revenue,source_revenue), '상품별 매출 검산 실패'
assert int(item_kpi['n_rows'].sum())==len(sales), '상품별 집계 건수 검산 실패'
print('월별·매장별 KPI:');display(store_month)
print('상품별 KPI:');display(item_kpi)
print(f'집계 전후 매출 검증: 원본 {source_revenue:,.0f} = 매장·월 {store_revenue:,.0f} = 상품 {item_revenue:,.0f}원')

월별·매장별 KPI:


,store_id,2025-04,2025-05,2025-06
0,S01,6834000.0,7243500.0,5884000.0
1,S02,7531500.0,6768500.0,7996500.0
2,S03,7424500.0,4548000.0,7297500.0
3,S04,6471500.0,7105000.0,6744500.0
4,S05,8681500.0,4102000.0,4960000.0
5,S06,9156000.0,5095500.0,6796500.0


상품별 KPI:


,item,avg_unit_price,total_revenue,n_rows
0,쿠키,6878.082192,26690000.0,365
1,샌드위치,6555.405405,26297000.0,370
2,케이크,6859.331476,23788500.0,359
3,식빵,6554.929577,22997500.0,355
4,크루아상,6946.268657,20867500.0,335


집계 전후 매출 검증: 원본 120,640,500 = 매장·월 120,640,500 = 상품 120,640,500원


## 5. 저장(Save) — CSV와 Parquet 크기·자료형 비교

두 가지 KPI를 Parquet으로 저장하고, 같은 정제 데이터를 CSV·Parquet 양쪽에 저장해 실제 바이트 크기를 비교한다. **이 셀에서 `pyarrow` 미설치 메시지가 나오면 설치 후 다시 실행해야 필수 제출이 완료된다.**

설치 필요 시 별도 코드 셀에서 `%pip install pyarrow`를 실행하고 커널을 다시 시작한 뒤 전체 셀을 다시 실행한다.

In [7]:
csv_path=OUT_DIR/'bakery_clean.csv'
clean_pq=OUT_DIR/'bakery_clean.parquet'
store_pq=OUT_DIR/'bakery_store_month.parquet'
item_pq=OUT_DIR/'bakery_item_kpi.parquet'

bakery_clean.to_csv(csv_path,index=False,encoding='utf-8-sig')
print('정제 CSV 저장:',csv_path, f'({csv_path.stat().st_size:,} bytes)')

parquet_ok=importlib.util.find_spec('pyarrow') is not None
if not parquet_ok:
    print('필수 확인: pyarrow가 설치되어 있지 않아 Parquet 파일은 아직 생성되지 않았습니다.')
    print('노트북 새 셀에서 %pip install pyarrow 실행 → 커널 재시작 → 모두 실행하세요.')
else:
    bakery_clean.to_parquet(clean_pq,index=False)
    store_month.to_parquet(store_pq,index=False)
    item_kpi.to_parquet(item_pq,index=False)
    pq_bytes=clean_pq.stat().st_size
    csv_bytes=csv_path.stat().st_size
    print(f'크기 비교: CSV {csv_bytes:,} bytes / Parquet {pq_bytes:,} bytes')
    print(f'이번 데이터의 Parquet/CSV 비율: {pq_bytes/csv_bytes:.2%}')
    store_loaded=pd.read_parquet(store_pq)
    item_loaded=pd.read_parquet(item_pq)
    clean_loaded=pd.read_parquet(clean_pq)
    assert store_loaded.shape==store_month.shape
    assert item_loaded.shape==item_kpi.shape
    assert clean_loaded.shape==bakery_clean.shape
    assert pd.api.types.is_datetime64_any_dtype(clean_loaded['date']), '날짜 dtype 보존 실패'
    print('Parquet 3개 저장·재로드·shape·날짜 자료형 검증 완료')

정제 CSV 저장: c:\Users\shini\Downloads\D009_%EC%98%A4%EB%8A%98%EC%A0%9C%EC%B6%9C_%EC%A4%80%EB%B9%84%EB%AC%B6%EC%9D%8C\D009\outputs\bakery_clean.csv (91,863 bytes)
크기 비교: CSV 91,863 bytes / Parquet 9,887 bytes
이번 데이터의 Parquet/CSV 비율: 10.76%
Parquet 3개 저장·재로드·shape·날짜 자료형 검증 완료


## 6. 결정 로그(Decision Log) — 선택과 근거

실제 관측된 처리 건수를 함께 기록한다. 정제 규칙은 이번 **실습용 분석 시나리오**를 위한 판단이며, 실거래 데이터에서는 증빙에 따른 재판단이 필요하다.

In [8]:
store_strip_n=int(bakery_raw['store_id'].astype('string').ne(bakery_raw['store_id'].astype('string').str.strip()).sum())
item_alias_n=int(bakery_raw['item'].eq('케익').sum())
q50_n=int(bakery_clean['quantity'].eq(50).sum())
postdup_n=int(bakery_clean.duplicated().sum())
decisions=pd.DataFrame([
    ('1. 원본 중복',f'{dedup_n}행 제거','교육용 가정: 동일 행을 1회만 집계. 거래 ID 부재로 실제 거래의 동일성은 미검증'),
    ('2. 매장 ID',f'양끝 공백 {store_strip_n}행','매장 표기가 다르면 매장별 매출이 분리됨'),
    ('3. 상품명',f'케익 표기 {item_alias_n}행','케익과 케이크는 동일 범주로 정의'),
    ('4. 날짜',f'날짜 파싱 실패 {parse_fail}행','형식 혼재를 datetime으로 변환하고 실패 시 NaT 기록'),
    ('5. 환불 의심',f'{refund_n}행 별도 보존','음수는 구매 매출 KPI에서 제외하되 감사·환불 분석을 위해 남김'),
    ('6. 매출 결측',f'{missing_n}행 제외','총매출을 근거 없이 평균으로 채우지 않음'),
    ('7. 수량 이상치',f'정제본에서 50개 수량 {q50_n}행 유지','정상 대량 주문 가능성이 있어 일괄 삭제하지 않음'),
    ('8. 표준화 후 동일 행',f'{postdup_n}행 재삭제하지 않음','표기·날짜 통일로 새로 일치해 보일 수 있으나 거래 ID가 없어 실제 중복 확정 불가'),
],columns=['결정','결과','근거'])
display(decisions)

,결정,결과,근거
0,1. 원본 중복,321행 제거,교육용 가정: 동일 행을 1회만 집계. 거래 ID 부재로 실제 거래의 동일성은 미검증
1,2. 매장 ID,양끝 공백 10행,매장 표기가 다르면 매장별 매출이 분리됨
2,3. 상품명,케익 표기 8행,케익과 케이크는 동일 범주로 정의
3,4. 날짜,날짜 파싱 실패 0행,형식 혼재를 datetime으로 변환하고 실패 시 NaT 기록
4,5. 환불 의심,1행 별도 보존,음수는 구매 매출 KPI에서 제외하되 감사·환불 분석을 위해 남김
5,6. 매출 결측,58행 제외,총매출을 근거 없이 평균으로 채우지 않음
6,7. 수량 이상치,정제본에서 50개 수량 277행 유지,정상 대량 주문 가능성이 있어 일괄 삭제하지 않음
7,8. 표준화 후 동일 행,5행 재삭제하지 않음,표기·날짜 통일로 새로 일치해 보일 수 있으나 거래 ID가 없어 실제 중복 확정 불가


## 7. 실행 결과로 보고서/README 생성

보고서의 5개 항목: **① 데이터 개요 ② 발견한 품질 문제 ③ 처리 결정·근거 ④ 주요 KPI ⑤ 한계와 후속 작업**. 출력의 수치가 실제 계산값과 일치하는지 검토한다.

In [9]:
top_store=store_month.melt(id_vars='store_id',var_name='month',value_name='total_revenue').sort_values('total_revenue',ascending=False).iloc[0]
top_item=item_kpi.iloc[0]
missing_highest=quality_before.sort_values('missing_pct',ascending=False).iloc[0]
outlier_highest=(quality_before.dropna(subset=['outlier_pct_iqr']).sort_values('outlier_pct_iqr',ascending=False).iloc[0])
observed_dates=sorted(sales['date'].dt.strftime('%Y-%m-%d').unique())
file_compare=(f'CSV **{csv_path.stat().st_size:,} bytes** / Parquet **{clean_pq.stat().st_size:,} bytes**, '
              f'Parquet/CSV **{clean_pq.stat().st_size/csv_path.stat().st_size:.2%}**') if parquet_ok else (
              f'CSV **{csv_path.stat().st_size:,} bytes** / Parquet **미생성(로컬에서 pyarrow 설치 후 재실행 필요)**')
report_md=f"""# D009 빵셀러 데이터 정제·집계 보고서

## 1. 데이터 개요
- 출처: 공식 AIFFEL 종합 실습 빵셀러 **가상 생성 데이터** (해당 노트북에서 로컬 생성한 CSV)
- 공식 시나리오 기간: 2025년 4~6월. 실제 데이터의 날짜는 **{', '.join(observed_dates)}**만 관측됨(연속 90일 데이터가 아님).
- 원본 **{len(bakery_raw):,}행**, 정제 완료 **{len(bakery_clean):,}행**.
- 분석 목적: 정제 후 월·매장별 매출 및 상품별 KPI 집계(환불 의심 기록 별도 보관).

## 2. 발견한 품질 문제
- 결측률 최상위: **{missing_highest['column']} {missing_highest['missing_pct']:.2f}%**.
- 완전 동일한 원본 행: **{dedup_n:,}행** (공식 데이터 생성기에서 의도적으로 추가한 4행 이외에도 일치 행 존재).
- IQR 이상치 비율 상위 수치형 열: **{outlier_highest['column']} {outlier_highest['outlier_pct_iqr']:.2f}%**.
- 매장 ID 양끝 공백 **{store_strip_n}행**, 상품명 `케익` **{item_alias_n}행**, 환불 의심 음수 매출 **{refund_n}행**.
- 날짜 변환 실패 **{parse_fail}행**. 정제 후 수량 50인 행 **{q50_n}행**(의심값으로 검토, 자동 삭제 안 함).

## 3. 처리 결정과 근거
1. 완전 동일 원본 행 **{dedup_n:,}행 제거**: 이번 실습의 중복 제거 규칙 적용. **실제 거래 ID가 없어 정상 반복 거래를 지웠을 위험**은 남는다.
2. `store_id` 양끝 공백 제거: 동일 매장별 집계 분리 방지.
3. `케익 → 케이크` 통일: 동일 상품 매출의 분산 방지.
4. `date_str` 날짜 변환(`format='mixed', errors='coerce'`): 실패는 `NaT`로 기록하고 별도 검사.
5. 음수 `revenue` **{refund_n}행**은 `refunds_bakery`에 보존: 실습상 구매 매출에서 제외하되 임의로 삭제하지 않음.
6. `revenue` 결측 **{missing_n}행 제외**: 매출 합계의 임의 추정 금지.
7. `quantity=50` 행 **{q50_n}행**은 제거하지 않음: 실제 대량 구매 가능성.
8. 표준화 후 동일하게 보이는 **{postdup_n}행**은 추가 삭제하지 않음: 거래 ID를 확인할 수 없기 때문.

## 4. 주요 KPI 결과
- 월·매장 기준 최고 매출: **{top_store['store_id']} / {top_store['month']} / {top_store['total_revenue']:,.0f}원**.
- 상품 매출 1위: **{top_item['item']} / {top_item['total_revenue']:,.0f}원**.
- 정제된 환불 제외 매출 합계: **{source_revenue:,.0f}원**.
- 매장·월별 KPI 합계와 상품별 KPI 합계: **각각 {store_revenue:,.0f}원, {item_revenue:,.0f}원** (원본 정제 매출과 대조 통과).
- 정제본 저장 파일 크기: {file_compare}.

## 5. 한계와 후속 작업
- **거래 ID가 없어 원본 동일 행 {dedup_n}건이 실제 재거래인지 검증 불가.** 실무 적용 시 주문·영수증 ID와 시각 추가 필요.
- 4~6월 전체 일자 데이터가 아니라 일부 날짜에 집중된 합성 데이터이므로 일별 추세나 실제 영업성과로 일반화하지 않는다.
- 음수 매출을 환불 의심으로 분류했으나 실제 환불 증빙은 없다. 구매 매출과 환불 반영 순매출은 구분해야 한다.
- 수량 이상치는 삭제하지 않았다. 상품별 단가는 판매 행의 `unit_price` **단순 평균**이며 수량 가중평균과 다르다.
- CSV/Parquet 크기 차이는 실측값이며 데이터 구조·압축 방식·크기에 따라 달라진다.

### 검증 및 재현
- 원본 행수 재조정·결측·음수매출·문자열 정제 검사: **통과**.
- KPI별 매출 합계·상품별 행 수 검산: **통과**.
- Parquet 출력·재읽기 검증: **{'통과' if parquet_ok else '미완료: pyarrow 설치 후 전체 재실행 필요'}**.
- AI 사용: 코드 구조·검증·문서 초안 보조. 수업 과제의 업무 가정과 제출 적합성은 학습자가 최종 확인.
"""
(BASE/'README.md').write_text(report_md,encoding='utf-8')
print('실제 계산값으로 README 저장:',BASE/'README.md')
display(Markdown(report_md))

실제 계산값으로 README 저장: c:\Users\shini\Downloads\D009_%EC%98%A4%EB%8A%98%EC%A0%9C%EC%B6%9C_%EC%A4%80%EB%B9%84%EB%AC%B6%EC%9D%8C\D009\README.md


# D009 빵셀러 데이터 정제·집계 보고서

## 1. 데이터 개요
- 출처: 공식 AIFFEL 종합 실습 빵셀러 **가상 생성 데이터** (해당 노트북에서 로컬 생성한 CSV)
- 공식 시나리오 기간: 2025년 4~6월. 실제 데이터의 날짜는 **2025-04-01, 2025-05-15, 2025-06-20**만 관측됨(연속 90일 데이터가 아님).
- 원본 **2,164행**, 정제 완료 **1,784행**.
- 분석 목적: 정제 후 월·매장별 매출 및 상품별 KPI 집계(환불 의심 기록 별도 보관).

## 2. 발견한 품질 문제
- 결측률 최상위: **revenue 2.77%**.
- 완전 동일한 원본 행: **321행** (공식 데이터 생성기에서 의도적으로 추가한 4행 이외에도 일치 행 존재).
- IQR 이상치 비율 상위 수치형 열: **revenue 17.30%**.
- 매장 ID 양끝 공백 **10행**, 상품명 `케익` **8행**, 환불 의심 음수 매출 **1행**.
- 날짜 변환 실패 **0행**. 정제 후 수량 50인 행 **277행**(의심값으로 검토, 자동 삭제 안 함).

## 3. 처리 결정과 근거
1. 완전 동일 원본 행 **321행 제거**: 이번 실습의 중복 제거 규칙 적용. **실제 거래 ID가 없어 정상 반복 거래를 지웠을 위험**은 남는다.
2. `store_id` 양끝 공백 제거: 동일 매장별 집계 분리 방지.
3. `케익 → 케이크` 통일: 동일 상품 매출의 분산 방지.
4. `date_str` 날짜 변환(`format='mixed', errors='coerce'`): 실패는 `NaT`로 기록하고 별도 검사.
5. 음수 `revenue` **1행**은 `refunds_bakery`에 보존: 실습상 구매 매출에서 제외하되 임의로 삭제하지 않음.
6. `revenue` 결측 **58행 제외**: 매출 합계의 임의 추정 금지.
7. `quantity=50` 행 **277행**은 제거하지 않음: 실제 대량 구매 가능성.
8. 표준화 후 동일하게 보이는 **5행**은 추가 삭제하지 않음: 거래 ID를 확인할 수 없기 때문.

## 4. 주요 KPI 결과
- 월·매장 기준 최고 매출: **S06 / 2025-04 / 9,156,000원**.
- 상품 매출 1위: **쿠키 / 26,690,000원**.
- 정제된 환불 제외 매출 합계: **120,640,500원**.
- 매장·월별 KPI 합계와 상품별 KPI 합계: **각각 120,640,500원, 120,640,500원** (원본 정제 매출과 대조 통과).
- 정제본 저장 파일 크기: CSV **91,863 bytes** / Parquet **9,887 bytes**, Parquet/CSV **10.76%**.

## 5. 한계와 후속 작업
- **거래 ID가 없어 원본 동일 행 321건이 실제 재거래인지 검증 불가.** 실무 적용 시 주문·영수증 ID와 시각 추가 필요.
- 4~6월 전체 일자 데이터가 아니라 일부 날짜에 집중된 합성 데이터이므로 일별 추세나 실제 영업성과로 일반화하지 않는다.
- 음수 매출을 환불 의심으로 분류했으나 실제 환불 증빙은 없다. 구매 매출과 환불 반영 순매출은 구분해야 한다.
- 수량 이상치는 삭제하지 않았다. 상품별 단가는 판매 행의 `unit_price` **단순 평균**이며 수량 가중평균과 다르다.
- CSV/Parquet 크기 차이는 실측값이며 데이터 구조·압축 방식·크기에 따라 달라진다.

### 검증 및 재현
- 원본 행수 재조정·결측·음수매출·문자열 정제 검사: **통과**.
- KPI별 매출 합계·상품별 행 수 검산: **통과**.
- Parquet 출력·재읽기 검증: **통과**.
- AI 사용: 코드 구조·검증·문서 초안 보조. 수업 과제의 업무 가정과 제출 적합성은 학습자가 최종 확인.


# D009 빵셀러 데이터 정제·집계 보고서

## 1. 데이터 개요
- 출처: 공식 AIFFEL 종합 실습 빵셀러 **가상 생성 데이터** (해당 노트북에서 로컬 생성한 CSV)
- 공식 시나리오 기간: 2025년 4~6월. 실제 데이터의 날짜는 **2025-04-01, 2025-05-15, 2025-06-20**만 관측됨(연속 90일 데이터가 아님).
- 원본 **2,164행**, 정제 완료 **1,784행**.
- 분석 목적: 정제 후 월·매장별 매출 및 상품별 KPI 집계(환불 의심 기록 별도 보관).

## 2. 발견한 품질 문제
- 결측률 최상위: **revenue 2.77%**.
- 완전 동일한 원본 행: **321행** (공식 데이터 생성기에서 의도적으로 추가한 4행 이외에도 일치 행 존재).
- IQR 이상치 비율 상위 수치형 열: **revenue 17.30%**.
- 매장 ID 양끝 공백 **10행**, 상품명 `케익` **8행**, 환불 의심 음수 매출 **1행**.
- 날짜 변환 실패 **0행**. 정제 후 수량 50인 행 **277행**(의심값으로 검토, 자동 삭제 안 함).

## 3. 처리 결정과 근거
1. 완전 동일 원본 행 **321행 제거**: 이번 실습의 중복 제거 규칙 적용. **실제 거래 ID가 없어 정상 반복 거래를 지웠을 위험**은 남는다.
2. `store_id` 양끝 공백 제거: 동일 매장별 집계 분리 방지.
3. `케익 → 케이크` 통일: 동일 상품 매출의 분산 방지.
4. `date_str` 날짜 변환(`format='mixed', errors='coerce'`): 실패는 `NaT`로 기록하고 별도 검사.
5. 음수 `revenue` **1행**은 `refunds_bakery`에 보존: 실습상 구매 매출에서 제외하되 임의로 삭제하지 않음.
6. `revenue` 결측 **58행 제외**: 매출 합계의 임의 추정 금지.
7. `quantity=50` 행 **277행**은 제거하지 않음: 실제 대량 구매 가능성.
8. 표준화 후 동일하게 보이는 **5행**은 추가 삭제하지 않음: 거래 ID를 확인할 수 없기 때문.

## 4. 주요 KPI 결과
- 월·매장 기준 최고 매출: **S06 / 2025-04 / 9,156,000원**.
- 상품 매출 1위: **쿠키 / 26,690,000원**.
- 정제된 환불 제외 매출 합계: **120,640,500원**.
- 매장·월별 KPI 합계와 상품별 KPI 합계: **각각 120,640,500원, 120,640,500원** (원본 정제 매출과 대조 통과).
- CSV와 Parquet 실측 크기 비교: 위 **저장(Save)** 코드 셀의 실행 결과 및 실행 후 자동 갱신되는 `README.md` 참조.

## 5. 한계와 후속 작업
- **거래 ID가 없어 원본 동일 행 321건이 실제 재거래인지 검증 불가.** 실무 적용 시 주문·영수증 ID와 시각 추가 필요.
- 4~6월 전체 일자 데이터가 아니라 일부 날짜에 집중된 합성 데이터이므로 일별 추세나 실제 영업성과로 일반화하지 않는다.
- 음수 매출을 환불 의심으로 분류했으나 실제 환불 증빙은 없다. 구매 매출과 환불 반영 순매출은 구분해야 한다.
- 수량 이상치는 삭제하지 않았다. 상품별 단가는 판매 행의 `unit_price` **단순 평균**이며 수량 가중평균과 다르다.
- CSV/Parquet 크기 차이는 실측값이며 데이터 구조·압축 방식·크기에 따라 달라진다.

### 검증 및 재현
- 원본 행수 재조정·결측·음수매출·문자열 정제 검사: **통과**.
- KPI별 매출 합계·상품별 행 수 검산: **통과**.
- Parquet 출력·재읽기 검증: **제출 환경에서 저장 셀을 실행한 뒤 성공 여부를 확인해야 함**.
- AI 사용: 코드 구조·검증·문서 초안 보조. 수업 과제의 업무 가정과 제출 적합성은 학습자가 최종 확인.

> 제출 전 확인: 전체 셀 실행 후 Parquet 파일 2개가 실제로 생성됐는지 검증해야 합니다. 위 수치는 첨부된 공식 합성 데이터에서 계산했습니다.


In [10]:
import pandas as pd

# 저장한 상품별 KPI 파일 읽기
item_kpi = pd.read_parquet(
    "outputs/bakery_item_kpi.parquet"
)

# 첫 5행을 표로 보여주기
display(item_kpi.head())

,item,avg_unit_price,total_revenue,n_rows
0,쿠키,6878.082192,26690000.0,365
1,샌드위치,6555.405405,26297000.0,370
2,케이크,6859.331476,23788500.0,359
3,식빵,6554.929577,22997500.0,355
4,크루아상,6946.268657,20867500.0,335


In [11]:
# 1. 월별·매장별 KPI 파일 읽기
store_kpi = pd.read_parquet(
    "outputs/bakery_store_month.parquet"
)

# 2. 결과 확인
print("월별·매장별 매출:")
display(store_kpi)

# 3. 상품별 매출 총합 확인
print("상품별 매출 합계:")
print(item_kpi["total_revenue"].sum())

# 4. 저장된 파일의 크기와 자료형 확인
print("월별·매장별 KPI 크기:", store_kpi.shape)
print("상품별 KPI 크기:", item_kpi.shape)
print("상품별 KPI 자료형:")
print(item_kpi.dtypes)

월별·매장별 매출:


,store_id,2025-04,2025-05,2025-06
0,S01,6834000.0,7243500.0,5884000.0
1,S02,7531500.0,6768500.0,7996500.0
2,S03,7424500.0,4548000.0,7297500.0
3,S04,6471500.0,7105000.0,6744500.0
4,S05,8681500.0,4102000.0,4960000.0
5,S06,9156000.0,5095500.0,6796500.0


상품별 매출 합계:
120640500.0
월별·매장별 KPI 크기: (6, 4)
상품별 KPI 크기: (5, 4)
상품별 KPI 자료형:
item               string
avg_unit_price    float64
total_revenue     float64
n_rows              int64
dtype: object
